In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP05 - TP Gov't Int
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   Number of third party relationships or engagements which required 1. the arrangement 
   of government issued approval, license and/or permits, etc. OR 2. the third party 
   to apply or interact with a PO on behalf of TD?
   
   LOGIC & ARCHITECTURE SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. RISK FLAG LOGIC: Flags engagements as high risk if KPI_Number = '8.2' AND 
      Value = 'YES'.
   3. EXPANDED COMMA EXCEPTION HANDLING: Protects 6 known unit names containing internal 
      commas by temporarily replacing them with '[COMMA]' so the split function doesn't 
      shred them.
   4. FLATTEN LOBs: Uses LATERAL VIEW explode(split(...)) to expand the comma-separated 
      owning_lob and lob_using lists into individual rows. Restores the '[COMMA]' back 
      to a real ',' immediately after.
   5. SAFE MAPPING: Maps expanded LOBs to TPRM_Units using standard LIKE syntax 
      ('%' || String || '%') to avoid regex failures on special characters ('&', '-').
   6. NAME BRIDGE: The mapping table stores the AU Name in the Assessable_Unit_ID 
      column. This logic bridges that String Name to the Master List's AU_Name to 
      retrieve the true Numeric BusinessID.
   7. FINAL OUTPUT: Strict 6-column structure, counting the distinct numerical 
      engagements mapped to the AU and converting NULL sums to '0'.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    -- 2. Grab valid TPRM Mapping Strings and block blank/null strings
    -- Note: Assessable_Unit_ID actually contains the string Name, not the numeric ID
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_AU_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

Flagged_Engagements AS (
    -- 3. Filter the TP data strictly based on KPI 8.2 = YES, and apply Expanded Comma Exceptions
    SELECT DISTINCT
        EngagementNumber,
        
        -- =========================================================================
        -- EXPANDED EXCEPTION DICTIONARY: Protect all known LOBs with internal commas
        -- =========================================================================
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
            
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using
            
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
      AND TRIM(KPI_Number) = '8.2' 
      AND UPPER(TRIM(Value)) = 'YES'
),

Flattened_LOBs AS (
    -- 4. FLATTEN RULE: Split by commas, expand into individual rows, and restore protected commas
    SELECT 
        EngagementNumber, 
        -- RESTORE: Put the actual commas back so it matches the Mapping Table!
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM Flagged_Engagements
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Engagements_By_AU AS (
    -- 5. Bridge mappings to Master List and count the matches
    SELECT 
        mast.BusinessID,
        -- DEDUPLICATION: Count distinct engagements to avoid double-counting expanded rows
        COUNT(DISTINCT f.EngagementNumber) AS Match_Count
    FROM Flattened_LOBs f
    -- Safe Mapping Join
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
    -- Name Bridge Join
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_AU_Name)) = UPPER(TRIM(mast.AU_Name))
    GROUP BY mast.BusinessID
)

-- 6. Final Output: Strict 6-column structure anchored to Master AU List
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP05' AS QuestionID,               
    COALESCE(CAST(e.Match_Count AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.BusinessID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP05 - Gov't Interaction Traceability
   
   PURPOSE: Verifies the KPI flag logic, prints the full comma-separated LOB 
   strings vs the exploded chunks (with EXPANDED comma protection), and displays 
   the Master AU lookup status to troubleshoot mapping drops.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS Master_Numeric_ID,
        TRIM(Business_Operational_Unit_Name) AS Master_AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
),

Filtered_TP_Data AS (
    -- Extract raw data, apply the KPI flag, and protect the commas in the LOB strings
    SELECT 
        p.EngagementNumber,
        p.KPI_Number,
        p.Value,
        p.owning_lob AS Raw_Col_K,
        p.lob_using AS Raw_Col_L,
        
        -- Expanded Comma Dictionary
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(p.owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
            
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(p.lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using
            
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 p
    WHERE p.EngagementNumber IS NOT NULL
      AND TRIM(p.KPI_Number) = '8.2' 
      AND UPPER(TRIM(p.Value)) = 'YES'
),

Flattened AS (
    -- Explode the LOBs into individual rows and restore commas
    SELECT 
        f.EngagementNumber,
        f.KPI_Number,
        f.Value,
        f.Raw_Col_K,
        f.Raw_Col_L,
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Exploded_Chunk
    FROM Filtered_TP_Data f
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
)

SELECT DISTINCT
    f.EngagementNumber,
    
    -- KPI Traceability
    f.KPI_Number,
    f.Value,
    
    -- LOB Flattening Traceability
    f.Raw_Col_K AS Original_String_From_DB_Col_K,
    f.Raw_Col_L AS Original_String_From_DB_Col_L,
    f.Exploded_Chunk AS Final_Restored_Chunk,
    
    -- Mapping & Bridging Traceability
    map.TPRM_Units AS Matched_Mapping_String,
    map.Assessable_Unit_ID AS Mapping_Table_AU_Name,
    mast.Master_Numeric_ID AS Bridged_Master_ID,
    CASE WHEN mast.Master_Numeric_ID IS NULL THEN 'MISSING FROM MASTER LIST' ELSE 'SUCCESSFULLY BRIDGED' END AS Master_List_Status
    
FROM Flattened f
-- Safe Mapping Join
LEFT JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map 
    ON UPPER(f.Exploded_Chunk) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
-- Name Bridge Join
LEFT JOIN Master_AUs mast 
    ON UPPER(TRIM(map.Assessable_Unit_ID)) = UPPER(TRIM(mast.Master_AU_Name))
ORDER BY f.EngagementNumber;